In [ ]:
# H2-KN02 (spec v1.2 "Held-out set 2: step 3 embedding on Kaggle", 2026-11-23): bge-m3 vectors (pinned revision,
# float32, normalised, batch 16, as P3-P4-KN01 for held-out 1) for every chunk text of the 26 primary setups on the 53
# held_out_2 documents and every held_out_2 query. Sealed split: --final, and only the files of dataset H2-KD02
# (scripts/h2_embed_bundle.py --final), each checked against BUNDLE.json and this commit's split.
# Needs: GPU T4, Internet on (clone + model download), dataset H2-KD02 attached. No secrets. Estimated about 2.5 h.
COMMIT = ""  # the commit that holds this notebook, scripts/p3_embed.py --questions and the step 3 note
assert COMMIT, "set COMMIT before running"
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git checkout -q {COMMIT} && git log --oneline -1 && git status --porcelain | head

In [ ]:
# Install, copied unchanged from the held-out 1 embedding notebook (notebooks/kaggle/p4-kn01.ipynb, P3-P4-KN01).
!pip install -q docling==2.143.0 jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 google-genai==2.34.0
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Extra pins, copied unchanged from notebooks/kaggle/rob-kn01.ipynb: the study's stanza and langchain pins.
!grep -iE "^(stanza|langchain[a-z_-]*)==" /kaggle/working/repo/requirements.txt | tee /kaggle/working/extra_pins.txt
!pip install -q -r /kaggle/working/extra_pins.txt

In [ ]:
# Fail fast: the imports (as rob-kn01, with the embedding script), the pip freeze and the GPU.
!cd /kaggle/working/repo && python -c "import src.chunk.chunkers, scripts.p3_embed; print('imports OK')"
!pip freeze > /kaggle/working/h2-kn02_pip_freeze.txt
!grep -i -E "^(torch|transformers|sentence-transformers|tokenizers|numpy)==" /kaggle/working/h2-kn02_pip_freeze.txt
assert torch.cuda.is_available(), "turn the GPU on"

In [ ]:
# Copy H2-KD02 into the clone at its repo paths. The bundle must hold exactly the held_out_2 documents of this commit's
# pdfs/split.csv and 26 setups (13 settings x 2 parsers); every file must match BUNDLE.json by SHA-256.
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd

REPO, INPUT, TEMP = Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/temp/h2-kd02")
sys.path.insert(0, str(REPO))
record_path = next(INPUT.rglob("BUNDLE.json"), None)
assert record_path, "attach the dataset aman125/h2-kd02-heldout2 (BUNDLE.json not found under /kaggle/input)"
bundle = record_path.parent
if not (bundle / "data").is_dir():  # uploaded with --dir-mode zip and not unpacked by Kaggle
    for archive in bundle.glob("*.zip"):
        zipfile.ZipFile(archive).extractall(TEMP / archive.stem)
    bundle = next(TEMP.rglob("BUNDLE.json")).parent
record = json.loads(record_path.read_text())
split = pd.read_csv(REPO / "pdfs" / "split.csv")
held_out_2 = set(split.loc[split["split"] == "held_out_2", "doc_id"])
assert len(held_out_2) == 53 and set(record["documents"]) == held_out_2, "H2-KD02 documents differ from held_out_2"
assert record["setups"] == 26 and len(record["settings"]) == 13 and record["parsers"] == ["docling", "pdfplumber"]
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
for entry in record["files"]:
    source, target = bundle / entry["path"], REPO / entry["path"]
    in_split = entry["path"].startswith(("data/heldout2/", "data/embeddings/")) or Path(entry["path"]).stem in held_out_2
    assert in_split and sha(source) == entry["sha256"], entry["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
chunk_files = list((REPO / "data" / "chunks").rglob("*.json"))
assert {path.stem for path in chunk_files} == held_out_2 and len(chunk_files) == 26 * 53, "chunk files differ"
print(len(record["files"]), "files copied and checked |", len(chunk_files), "chunk files |", record["chunk_texts"],
      "chunk texts |", record["questions"], "questions | cached", record["cached"])

In [ ]:
# Fail fast: imports, the pinned model loads on the GPU, one text embeds with the expected shape and norm.
from scripts.p3_embed import load_model
from src.eval.embed import EMBEDDER
model = load_model(EMBEDDER, "cuda")
probe = model.encode(["probe"], normalize_embeddings=True, convert_to_numpy=True)
assert probe.shape == (1, EMBEDDER["dimension"]) and abs(float((probe ** 2).sum()) - 1) < 1e-4
print("model OK |", EMBEDDER["name"], EMBEDDER["revision"][:7], "| dimension", probe.shape[1])
del model

In [ ]:
# bge-m3 vectors, run source H2-KN02 -> run record P3-H2-KN02-...; texts in the H2-KD02-cached part are skipped.
DOCS, SETTINGS = " ".join(sorted(held_out_2)), " ".join(record["settings"])
!cd /kaggle/working/repo && python -m scripts.p3_embed --final --docs {DOCS} --settings {SETTINGS} --questions data/heldout2/questions.jsonl --cache data/embeddings/bge-m3 --name H2-KN02-heldout2 --machine kaggle --run-source H2-KN02

In [ ]:
# Check, then one zip of the new vector part, the run record and the pip freeze, at repo paths (unzip at the repo root).
from src.eval.embed import chunk_texts, text_key
CACHE = REPO / "data" / "embeddings" / "bge-m3"
new = json.loads((CACHE / "H2-KN02-heldout2.keys.json").read_text())
cached = set(json.loads((CACHE / "H2-KD02-cached.keys.json").read_text()))
questions = {text_key(json.loads(line)["question"]) for line in (REPO / "data/heldout2/questions.jsonl").read_text().splitlines()}
wanted = set(chunk_texts(REPO / "data" / "chunks", held_out_2, record["settings"])) | questions
assert not set(new) & cached and set(new) | cached >= wanted, "vectors missing or a key held twice"
files = [CACHE / "H2-KN02-heldout2.npy", CACHE / "H2-KN02-heldout2.keys.json"] + sorted((REPO / "runs").glob("P3-H2-KN02-*.json"))
with zipfile.ZipFile("/kaggle/working/h2-kn02_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(REPO))
    archive.write("/kaggle/working/h2-kn02_pip_freeze.txt", "data/external/kaggle_out/H2-KN02/h2-kn02_pip_freeze.txt")
print(len(new), "new vectors |", len(cached), "cached |", len(wanted), "texts needed | zipped:", [path.name for path in files])
!ls -la /kaggle/working